In [ ]:
import os
import math

import ee
import geemap
import pandas as pd

# ---------------------------
# 1. Initialize Earth Engine
# ---------------------------
ee.Authenticate()
ee.Initialize(project="wetland-segmentation")

# ---------------------------
# 2. Parameters
# ---------------------------
EXCEL_PATH = "Ontario-Wetland.xlsx"
SCALE = 10  # Sentinel-2 resolution (10m)
TILE_PIXELS = 750  # For lower latitude
# TILE_PIXELS = 900  # For higher latitude
OUTPUT_DIR_BASE = "2_sat_imgs"

# os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# 3. Read Excel
# ---------------------------
df = pd.read_excel(EXCEL_PATH)

# ---------------------------
# 4. Loop over regions
# ---------------------------
for _, row in df.iterrows():

    region_id = str(row["ID"]).zfill(3)
    lat = row["Latitude"]
    lon = row["Longitude"]

    # Determine tile count
    n_tiles = None
    split = None

    if row["Usage"] == "Train":
        n_tiles = int(row["tiles(Train)"])
        split = "train"
    elif row["Usage"] == "Test":
        n_tiles = int(row["tiles(Test)"])
        split = "test"

    print(f"\nProcessing {region_id}")

    # ----------------------------------------
    # Compute dynamic sizes
    # ----------------------------------------
    tile_side = TILE_PIXELS * SCALE
    grid_size = int(math.sqrt(n_tiles))
    total_side_length = grid_size * tile_side
    half_total = total_side_length / 2
    sub_side = tile_side

    print(f"Grid size: {grid_size} x {grid_size}")
    print(f"Sub-tile side (meters): {sub_side}")

    # ----------------------------------------
    # Project to metric CRS (Web Mercator)
    # ----------------------------------------
    proj = ee.Projection("EPSG:3857")

    center_proj = ee.Geometry.Point([lon, lat]).transform(proj, 1)
    coords = center_proj.coordinates()
    cx = coords.get(0)
    cy = coords.get(1)

    # ----------------------------------------
    # Get Sentinel-2 image
    # ----------------------------------------
    roi_full = center_proj.buffer(half_total).bounds()
    print(f"roi_full: {roi_full.coordinates().getInfo()}")

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi_full)
        .filterDate("2021-06-15", "2021-09-15")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )

    print("Number of scenes:", collection.size().getInfo())

    image = collection.first()
    # image = collection.median()  # This gets worse

    if image is None:
        print("No image found.")
        continue

    image = image.select(["B2", "B3", "B4", "B8", "B11", "B12"])

    # ----------------------------------------
    # Generate N x N grid manually
    # ----------------------------------------
    tile_index = 1

    for i in range(grid_size):
        for j in range(grid_size):

            dx = (-half_total + sub_side/2) + i * sub_side
            dy = (-half_total + sub_side/2) + j * sub_side

            # Compute bounding box corners in projected CRS
            x_min = ee.Number(cx).add(dx).subtract(sub_side / 2)
            x_max = ee.Number(cx).add(dx).add(sub_side / 2)
            y_min = ee.Number(cy).add(dy).subtract(sub_side / 2)
            y_max = ee.Number(cy).add(dy).add(sub_side / 2)

            tile_roi_proj = ee.Geometry.Rectangle(
                [x_min, y_min, x_max, y_max],
                proj,
                False
            )

            tile_roi = tile_roi_proj.transform("EPSG:4326", 1)
            print(f"tile_roi: {tile_roi.coordinates().getInfo()}")

            output_path = os.path.join(
                OUTPUT_DIR_BASE,
                split,
                f"{split}_{region_id}_{str(tile_index).zfill(3)}.tif"
            )

            print(f"Exporting tile {tile_index}")

            # image = image.unmask(0)
            # image = image.clip(tile_roi)
            print(image.reduceRegion(
                reducer=ee.Reducer.count(),
                geometry=tile_roi,
                scale=10,
                maxPixels=1e9
            ).getInfo())


            geemap.ee_export_image(
                ee_object=image,
                filename=output_path,
                scale=SCALE,
                region=tile_roi,
                file_per_band=False,
                format="ZIPPED_GEO_TIFF",
                unzip=True,
                timeout=300,
            )

            tile_index += 1

print("\nAll exports completed.")